# Audit a Multi-Agent Debate Log with PACE
Paper: [arXiv:2611.17610](https://arxiv.org/abs/2611.17610) | PACE Studio: [sajjadi/pace-studio](https://huggingface.co/spaces/sajjadi/pace-studio) | Repo: [mahmoudsajjadi/pace](https://github.com/mahmoudsajjadi/pace)

In [ ]:
#@title Audit a multi-agent debate log with PACE
#@markdown Each record needs: item, agent, round, correct (plus prompt_tokens, completion_tokens if available).
USE_DEMO = True  #@param {type:"boolean"}
!pip -q install "git+https://github.com/mahmoudsajjadi/pace"

import json, pathlib, random, subprocess, sys
from IPython.display import Markdown, display

if USE_DEMO:
    # Synthetic log for illustration only; it does not come from a model run.
    random.seed(0)
    rows = []
    for i in range(40):
        p = random.choice([0.35, 0.55, 0.75])
        state = [random.random() < p for _ in range(5)]
        for rnd in (1, 2, 3):
            if rnd > 1:
                state = [(random.random() > 0.10) if s else (random.random() < 0.20) for s in state]
            for a, s in enumerate(state):
                rows.append({
                    "item": f"demo_{i:02d}",
                    "agent": f"agent_{a}",
                    "round": rnd,
                    "correct": bool(s),
                    "prompt_tokens": 300 + 250 * (rnd - 1),
                    "completion_tokens": 110,
                    "synthetic": True,
                })
    path = "demo_synthetic.jsonl"
    pathlib.Path(path).write_text("\n".join(json.dumps(r) for r in rows) + "\n")
else:
    from google.colab import files
    path = next(iter(files.upload()))

def pace(*args):
    out = subprocess.run([sys.executable, "-m", "pace", *args], capture_output=True, text=True)
    return out.stdout if out.returncode == 0 else "ERROR\n" + out.stderr

display(Markdown(pace("card", path)))
print(pace("dynamics", path, "--n-boot", "200"))